# Feature Transformation and Model Building

In [7]:
# IMPORTS
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_curve, auc, precision_recall_curve, average_precision_score

In [2]:
# DATA CURATION - sequencing and clinical data
rna_raw = pd.read_csv('../NSCLC Data/tpm_counts.csv')
clinical_raw = pd.read_csv('../NSCLC Data/clinical.csv')

clinical_w_rna = clinical_raw[clinical_raw['Case ID'].isin(rna_raw['Unnamed: 0'])]

# remove columns that are missing more than 50% of the data
rna_cols_removed = rna_raw.dropna(thresh=rna_raw.shape[0] * 0.5, axis=1)

knn_imputer = KNNImputer(n_neighbors=10)
rna_imputed = knn_imputer.fit_transform(rna_cols_removed.drop('Unnamed: 0', axis=1))

del(rna_raw, clinical_raw, rna_cols_removed, knn_imputer)

In [3]:
# Perform PCA

num_components = 10

pca = PCA(n_components=num_components)
X_pca = pca.fit_transform(rna_imputed)

print("Total explained variance ratio: {}".format(np.sum(pca.explained_variance_ratio_)))

Total explained variance ratio: 0.8610570750119386


In [5]:
y = clinical_w_rna.Recurrence.map({'yes': 1, 'no': 0}).values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

In [ ]:
# Define parameter grids
param_grid_knn = {'n_neighbors': [3, 5, 7, 9]}
param_grid_tree = {'max_depth': [None, 10, 20, 30]}
param_grid_forest = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}

# Perform grid search
grid_search_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy')
grid_search_tree = GridSearchCV(DecisionTreeClassifier(), param_grid_tree, cv=5, scoring='accuracy')
grid_search_forest = GridSearchCV(RandomForestClassifier(), param_grid_forest, cv=5, scoring='accuracy')
grid_search_svm = GridSearchCV(SVC(probability=True), param_grid_svm, cv=5, scoring='accuracy')

grid_search_knn.fit(X_train, y_train)
grid_search_tree.fit(X_train, y_train)
grid_search_forest.fit(X_train, y_train)
grid_search_svm.fit(X_train, y_train)

# Print best parameters and scores
print(f"Best parameters for k-NN: {grid_search_knn.best_params_}")
print(f"Best score for k-NN: {grid_search_knn.best_score_}")

print(f"Best parameters for Decision Tree: {grid_search_tree.best_params_}")
print(f"Best score for Decision Tree: {grid_search_tree.best_score_}")

print(f"Best parameters for Random Forest: {grid_search_forest.best_params_}")
print(f"Best score for Random Forest: {grid_search_forest.best_score_}")

print(f"Best parameters for SVM: {grid_search_svm.best_params_}")
print(f"Best score for SVM: {grid_search_svm.best_score_}")
